In [1]:
from migration import ddl_resolver
from migration.ddl_resolver import DdlResolver
from src.utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer, CurDecomposerWriter
from migration.cur.metadata import CurMetadataProcessor
from migration.cur.generator import CurPySparkGenerator
from src.paths import *

In [2]:
USERNAME

'dungp'

In [3]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_broker.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_address.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_einvoice_customer_address.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()["cur"]

print(f"File gốc tại: {input_file}")

File gốc tại: C:\Users\dungp\projects\datalake-script\dml\cur\cur_dim_address.sql


In [4]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = CurSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = CurDecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_address\processing_steps


In [5]:

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = CurMetadataProcessor(source_rules)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: ['OWNER_ID', 'ADDRESS_OWNER_TYPE', 'ADDRESS_TYPE']
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_address\metadata\cur_dim_address.yaml


In [6]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = CurPySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
🎉 Hoàn tất toàn bộ Pipeline!


In [7]:
ddl_resolver  = DdlResolver()

ddl_resolver.enrich(pipeline_config, model_type="1")

{'columns': [{'name': 'owner_id', 'type': 'VARCHAR(50)', 'remark': None},
  {'name': 'address_owner_type', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'address_type', 'type': 'VARCHAR(10)', 'remark': None},
  {'name': 'address_line_1', 'type': 'VARCHAR(100)', 'remark': None},
  {'name': 'address_line_2', 'type': 'VARCHAR(100)', 'remark': None},
  {'name': 'address_line_3', 'type': 'VARCHAR(100)', 'remark': None},
  {'name': 'address_line_4', 'type': 'VARCHAR(100)', 'remark': None},
  {'name': 'city', 'type': 'VARCHAR(255)', 'remark': None},
  {'name': 'state', 'type': 'VARCHAR(50)', 'remark': None},
  {'name': 'postcode', 'type': 'VARCHAR(5)', 'remark': None},
  {'name': 'country', 'type': 'VARCHAR(3)', 'remark': None},
  {'name': 'address_create_date', 'type': 'DATE', 'remark': None},
  {'name': 'address_update_date', 'type': 'DATE', 'remark': None},
  {'name': 'line_of_business', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'source_record_id', 'type': 'VARCHAR(50)', 'rem

In [8]:
template_dir = PROJECT_ROOT / "template" / "migration"
model_folders = [d for d in template_dir.iterdir() if d.is_dir() and d.name.startswith('model_')]

for model_folder in model_folders:
    model_name = model_folder.name
    print(model_name.split("_")[-1].lower())
    # print(f"Processing model: {model_name}")
    # enricher = DdlResolver(source_rules=source_rules)
    # ddl_context = enricher.enrich(pipeline_config, model_type=model_name.split()[-1].lower())

1
2a
2b
3
3a
3b
4
5a
5b
6
